In [1]:
import sys
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QVBoxLayout, QHBoxLayout, QWidget, QLineEdit, QPushButton, QTextBrowser, QLabel
)
from PyQt5.QtGui import QFont
from PyQt5.QtGui import QIcon
from PyQt5.QtCore import Qt
from sentence_transformers import SentenceTransformer, util
import speech_recognition as sr
import pyttsx3
import pandas as pd
import torch

class VoiceEnabledChatbot(QMainWindow):
    def __init__(self, dataset_path):
        super().__init__()
        self.setWindowIcon(QIcon("GCUFLogo.png"))  
        self.setWindowTitle("InfoSphere - A MultiModal Intelligent Assistant")
        self.setGeometry(200, 100, 800, 600)

        # Load dataset and model
        self.faq_path = dataset_path
        self.faq_data = pd.read_csv(self.faq_path)
        self.questions = self.faq_data['Question'].tolist()
        self.answers = self.faq_data['Answer'].tolist()
        self.model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
        self.embeddings = self.model.encode(self.questions, convert_to_tensor=True)
        
        # Load dataset and model
        #self.data = pd.read_csv(dataset_path)
        #self.questions = self.data['Question'].tolist()
        #self.answers = self.data['Answer'].tolist()
        #self.model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
        #self.embeddings = self.model.encode(self.questions, convert_to_tensor=True)

        # Initialize TTS engine
        self.tts_engine = pyttsx3.init()

        # Main container
        self.central_widget = QWidget()
        self.setCentralWidget(self.central_widget)
        self.layout = QVBoxLayout(self.central_widget)

        # User personalization
        self.user_name = None
        self.personalization_label = QLabel("Please enter your name to personalize your experience:")
        self.personalization_label.setFont(QFont("Arial", 12))
        self.layout.addWidget(self.personalization_label)

        self.name_input = QLineEdit()
        self.name_input.setFont(QFont("Arial", 12))
        self.name_input.setPlaceholderText("Enter your name and press Enter")
        self.name_input.returnPressed.connect(self.set_user_name)
        self.layout.addWidget(self.name_input)

        # Chat display area
        self.chat_display = QTextBrowser()
        self.chat_display.setFont(QFont("Arial", 12))
        self.chat_display.setStyleSheet("""
            QTextBrowser {
                background-color: #F9F9F9;
                border: 1px solid #CCC;
                padding: 10px;
                border-radius: 5px;
            }
        """)
        self.layout.addWidget(self.chat_display)

        # User input area
        self.input_field = QLineEdit()
        self.input_field.setPlaceholderText("Type your message here...")
        self.input_field.setFont(QFont("Arial", 12))
        self.input_field.returnPressed.connect(self.send_text_query)
        self.layout.addWidget(self.input_field)

        # Buttons in a single row
        self.button_layout = QHBoxLayout()

        self.send_button = QPushButton("✉️ Send")
        self.send_button.setFont(QFont("Arial", 10))
        self.send_button.setStyleSheet("background-color: #4CAF50; color: white; padding: 5px 10px; border-radius: 5px;")
        self.send_button.clicked.connect(self.send_text_query)
        self.button_layout.addWidget(self.send_button)

        self.voice_button = QPushButton("🎤 Speak")
        self.voice_button.setFont(QFont("Arial", 10))
        self.voice_button.setStyleSheet("background-color: #2196F3; color: white; padding: 5px 10px; border-radius: 5px;")
        self.voice_button.clicked.connect(self.handle_speak_button)
        self.button_layout.addWidget(self.voice_button)

        self.clear_button = QPushButton("🧹 Clear Chat")
        self.clear_button.setFont(QFont("Arial", 10))
        self.clear_button.setStyleSheet("background-color: #FF9800; color: white; padding: 5px 10px; border-radius: 5px;")
        self.clear_button.clicked.connect(self.clear_chat)
        self.button_layout.addWidget(self.clear_button)

        self.reset_button = QPushButton("🔄 Reset")
        self.reset_button.setFont(QFont("Arial", 10))
        self.reset_button.setStyleSheet("background-color: #00BFFF; color: white; padding: 5px 10px; border-radius: 5px;")
        self.reset_button.clicked.connect(self.reset_chat)
        self.button_layout.addWidget(self.reset_button)

        self.exit_button = QPushButton("❌ Exit")
        self.exit_button.setFont(QFont("Arial", 10))
        self.exit_button.setStyleSheet("background-color: #F44336; color: white; padding: 5px 10px; border-radius: 5px;")
        self.exit_button.clicked.connect(self.close)
        self.button_layout.addWidget(self.exit_button)

        self.layout.addLayout(self.button_layout)

        # Greeting message
        self.greeting_message = "Welcome! I'm InfoSphere, your personal intelligent assistant. How can I help you today?"
        self.add_message(self.greeting_message, is_user=False)
        self.speak_response(self.greeting_message)


        # Admin panel (initially hidden)
        self.admin_label = QLabel("Admin Panel: Add a New Question-Answer Pair")
        self.admin_label.setFont(QFont("Arial", 12))
        self.admin_label.hide()
        self.layout.addWidget(self.admin_label)

        self.new_question_input = QLineEdit()
        self.new_question_input.setPlaceholderText("Enter the new question...")
        self.new_question_input.setFont(QFont("Arial", 12))
        self.new_question_input.hide()
        self.layout.addWidget(self.new_question_input)

        self.new_answer_input = QLineEdit()
        self.new_answer_input.setPlaceholderText("Enter the corresponding answer...")
        self.new_answer_input.setFont(QFont("Arial", 12))
        self.new_answer_input.hide()
        self.layout.addWidget(self.new_answer_input)

        self.add_entry_button = QPushButton("Add Entry")
        self.add_entry_button.setFont(QFont("Arial", 10))
        self.add_entry_button.setStyleSheet("background-color: #4CAF50; color: white; padding: 5px 10px; border-radius: 5px;")
        self.add_entry_button.clicked.connect(self.add_new_entry_gui)
        self.add_entry_button.hide()
        self.layout.addWidget(self.add_entry_button)

        self.save_button = QPushButton("Save Dataset")
        self.save_button.setFont(QFont("Arial", 10))
        self.save_button.setStyleSheet("background-color: #00BFFF; color: white; padding: 5px 10px; border-radius: 5px;")
        self.save_button.clicked.connect(self.save_dataset_gui)
        self.save_button.hide()
        self.layout.addWidget(self.save_button)

        # Add question button to toggle admin panel
        self.show_admin_panel_button = QPushButton("Add Question")
        self.show_admin_panel_button.setFont(QFont("Arial", 10))
        self.show_admin_panel_button.setStyleSheet("background-color: #FFA500; color: white; padding: 5px 10px; border-radius: 5px;")
        self.show_admin_panel_button.clicked.connect(self.toggle_admin_panel)
        self.layout.addWidget(self.show_admin_panel_button)

    def toggle_admin_panel(self):
        """Toggle between the admin panel and chatbot interface."""
        if self.admin_label.isHidden():
            # Show admin panel
            self.admin_label.show()
            self.new_question_input.show()
            self.new_answer_input.show()
            self.add_entry_button.show()
            self.save_button.show()
            self.input_field.hide()
            self.show_admin_panel_button.setText("Back to Chat")
        else:
            # Hide admin panel
            self.admin_label.hide()
            self.new_question_input.hide()
            self.new_answer_input.hide()
            self.add_entry_button.hide()
            self.save_button.hide()
            self.input_field.show()
            self.show_admin_panel_button.setText("Add Question")

    def add_new_entry_gui(self):
        """Add a new question-answer pair to the dataset."""
        question = self.new_question_input.text().strip()
        answer = self.new_answer_input.text().strip()

        if not question or not answer:
            self.add_message("Error: Both question and answer fields must be filled.", is_user=False)
            return

        # Update dataset and embeddings
        new_embedding = self.model.encode(question, convert_to_tensor=True)
        self.embeddings = torch.cat([self.embeddings, new_embedding.unsqueeze(0)])

        # Add to dataset
        new_row = pd.DataFrame({"Question": [question], "Answer": [answer]})
        self.faq_data = pd.concat([self.faq_data, new_row], ignore_index=True)

        # Update in-memory lists
        self.questions.append(question)
        self.answers.append(answer)

        # Clear input fields
        self.new_question_input.clear()
        self.new_answer_input.clear()

        self.add_message("New entry added successfully!", is_user=False)

    def save_dataset_gui(self):
        """Save the updated dataset to a CSV file."""
        try:
            self.faq_data.to_csv(self.faq_path, index=False)
            self.add_message("Dataset saved successfully!", is_user=False)
        except Exception as e:
            self.add_message(f"Error saving dataset: {e}", is_user=False)

    def set_user_name(self):
        """Set the user's name and personalize the experience."""
        self.user_name = self.name_input.text().strip()
        if self.user_name:
            personalization_message = f"Hello {self.user_name}! How can I assist you today?"
            self.add_message(personalization_message, is_user=False)
            self.speak_response(personalization_message)
            self.personalization_label.hide()
            self.name_input.hide()

    def add_message(self, text, is_user):
        """Add a message to the chat area."""
        sender = self.user_name+":" if is_user else "AdmitEase:"
        message_format = f"""
            <div style="
                text-align: {'right' if is_user else 'left'};
                margin: 10px;
            ">
                <span style="
                    display: inline-block;
                    padding: 10px;
                    border-radius: 10px;
                    max-width: 75%;
                    background-color: {'#DCF8C6' if is_user else '#FFFFFF'};  
                    color: black;
                    font-size: 12pt;
                ">
                    <b>{sender}</b> {text}
                </span>
            </div>
        """
        self.chat_display.append(message_format)
        self.chat_display.verticalScrollBar().setValue(self.chat_display.verticalScrollBar().maximum())

    def send_text_query(self):
        """Handle sending a text query."""
        user_text = self.input_field.text().strip()
        if user_text:
            self.add_message(user_text, is_user=True)
            self.input_field.clear()
            bot_response = self.get_response(user_text)
            self.add_message(bot_response, is_user=False)
            self.speak_response(bot_response)

    def handle_speak_button(self):
        """Handle the color change for the speak button and process voice query."""
        self.voice_button.setStyleSheet("background-color: #FF0000; color: white; padding: 5px 10px; border-radius: 5px;")
        self.send_voice_query()
        self.voice_button.setStyleSheet("background-color: #2196F3; color: white; padding: 5px 10px; border-radius: 5px;")

    def send_voice_query(self):
        """Handle sending a voice query."""
        recognizer = sr.Recognizer()
        with sr.Microphone() as source:
            self.add_message("Listening for your query...", is_user=False)
            try:
                #recognizer.adjust_for_ambient_noise(source, duration=2)
                audio = recognizer.listen(source, timeout=10, phrase_time_limit=15)
                user_query = recognizer.recognize_google(audio)
                self.add_message(user_query, is_user=True)

                if user_query.lower() == "exit":
                    self.close()
                elif user_query.lower() in ["thank you", "thanks"]:
                    thank_you_message = "You're welcome! Happy to help."
                    self.add_message(thank_you_message, is_user=False)
                    self.speak_response(thank_you_message)
                else:
                    bot_response = self.get_response(user_query)
                    self.add_message(bot_response, is_user=False)
                    self.speak_response(bot_response)
            except sr.WaitTimeoutError:
                self.add_message("I didn't hear anything. Please try again.", is_user=False)
            except sr.UnknownValueError:
                self.add_message("Sorry, I couldn't understand that. Please try again.", is_user=False)
            except sr.RequestError as e:
                self.add_message(f"Error with the recognition service: {e}", is_user=False)

    def get_response(self, query):
        """Find the best response for the query."""
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        similarity_scores = util.pytorch_cos_sim(query_embedding, self.embeddings)
        best_match_idx = torch.argmax(similarity_scores).item()
        confidence = similarity_scores[0][best_match_idx].item()

        if confidence > 0.5:
            return self.answers[best_match_idx]
        return "Hmm, I couldn't find an exact answer. Could you try rephrasing your question?"

    def speak_response(self, text):
        """Speak the response using TTS."""
        self.tts_engine.say(text)
        self.tts_engine.runAndWait()

    def clear_chat(self):
        """Clear the chat area."""
        self.chat_display.clear()

    def reset_chat(self):
        """Reset the chat for a new user."""
        self.clear_chat()
        self.user_name = None
        self.personalization_label.show()
        self.name_input.clear()
        self.name_input.show()
        self.greeting_message = "Welcome! I'm AdmitEase, your intelligent assistant. How can I help you today?"
        self.add_message(self.greeting_message, is_user=False)
        self.speak_response(self.greeting_message)

if __name__ == "__main__":
    dataset_path = "university_faq_v2.csv"  
    app = QApplication(sys.argv)
    window = VoiceEnabledChatbot(dataset_path)
    window.show()
    sys.exit(app.exec_())


C:\Users\Z\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TypeError: 'float' object is not subscriptable